In [ ]:
# ============================================================
# Q2: Customer Segmentation using K-Means Clustering + PCA
# ============================================================

# -----------------------------
# 1. Data Preparation
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load dataset
df = pd.read_csv("q2_customers.csv")

# Display first five rows
print("First Five Rows:")
print(df.head())

# Display shape
print("\nDataset Shape:")
print(df.shape)

# Scale all features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

print("\nData Scaling Completed")

K-Means clustering uses distance (Euclidean distance) to group data points.

If features are on different scales, variables with larger values (like annual_spend) can dominate smaller-scale variables (like visits_per_month), leading to incorrect clusters.

StandardScaler ensures all features contribute equally by transforming them to a common scale with mean = 0 and standard deviation = 1.

In [ ]:
# -----------------------------
# 2. Choosing K — Elbow Method
# -----------------------------

wcss = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',
        random_state=42,
        n_init=10
    )
    kmeans.fit(scaled_data)
    wcss.append(kmeans.inertia_)

# Plot Elbow Method
plt.figure(figsize=(8,5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.show()

The elbow point is where the decrease in WCSS starts slowing down significantly.

This point represents the best balance between:

having compact clusters
avoiding too many unnecessary clusters

From the graph, the elbow appears at K = 4 (example — replace based on your actual graph).

Therefore, K = 4 is selected as the optimal number of clusters.

In [ ]:
# -----------------------------
# 3. K-Means Clustering
# -----------------------------

optimal_k = 4  # Replace if your elbow suggests another value

kmeans = KMeans(
    n_clusters=optimal_k,
    init='k-means++',
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(scaled_data)

print("\nCluster Labels Added Successfully")
print(df.head())

In [ ]:
# Print Cluster Centroids

centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=df.columns[:-1]
)

print("\nCluster Centroids:")
print(centroids)

Business Interpretation of Clusters

Example interpretation:

Cluster 0 → Young customers with low annual spend but frequent visits
Cluster 1 → High-spending loyal customers with large basket sizes
Cluster 2 → Infrequent customers with long gaps between visits
Cluster 3 → Moderate spenders with balanced shopping behavior

These insights help businesses create targeted marketing strategies for each customer group.



In [ ]:
# -----------------------------
# 4. PCA - Dimensionality Reduction
# -----------------------------

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

# Create PCA dataframe
pca_df = pd.DataFrame(
    data=pca_data,
    columns=['PC1', 'PC2']
)

pca_df['cluster'] = df['cluster']

# Explained Variance Ratio
print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

In [ ]:
# Feature Loadings

loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=df.columns[:-1]
)

print("\nPCA Feature Loadings:")
print(loadings)

Interpretation of PC1 and PC2

PC1 captures the strongest variation in customer behavior.

If annual_spend, basket_size, and visits_per_month have high positive loadings, PC1 may represent overall customer value.

PC2 captures the second strongest variation.

If days_since_last_visit has a strong loading, PC2 may represent customer inactivity or engagement level.

This helps simplify customer patterns into two major dimensions.

In [ ]:
# -----------------------------
# 5. Cluster Visualisation
# -----------------------------

plt.figure(figsize=(8,6))
sns.scatterplot(
    x='PC1',
    y='PC2',
    hue='cluster',
    data=pca_df,
    palette='Set2',
    s=100
)

plt.title("Customer Segmentation using PCA + K-Means")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

K-Means clustering successfully grouped customers into meaningful segments based on purchasing behavior.

Using PCA helped reduce complexity and made cluster visualization easier.

These clusters allow businesses to:

identify loyal high-value customers
re-engage inactive customers
improve personalised marketing strategies
optimize customer retention efforts

Thus, customer segmentation supports better business decision-making and targeted marketing.